In [1]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
import streamlit as st

In [26]:
load_dotenv()

True

In [27]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [28]:
# state
class ChatState(TypedDict):
  messages:Annotated[list[BaseMessage], add_messages]

In [29]:
# Node
def chat_node(state:ChatState):
  messages = state["messages"]

  response = llm.invoke(messages)
  return {'messages':[response]}


In [30]:
checkpoint = MemorySaver()

In [31]:
graph=StateGraph(ChatState)
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot=graph.compile(checkpointer=checkpoint)

In [32]:
# thread_id = "2"
# initial_state = {
#     'messages': [HumanMessage(content='What is my name?')]
# }

# config = {'configurable': {'thread_id': thread_id}}
# response = chatbot.invoke(initial_state, config=config)

In [35]:
thread_id = "1"

while True:
    user_message = input('Type here: ')

    print('User:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI:', response['messages'][-1].content)

User: My name is Harsh
AI: Nice to meet you, Harsh! Is there something I can help you with or would you like to chat?
User: WHat is my name?
AI: Your name is Harsh.
User: Exit
